* prompt -> ChatPromptTemplate 

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)                                 
SANDBOX = ROOT / "sandbox" / "w4" / "day03"

BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : C:\workspace\hanwha-agent


In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 사내 규정 질의 응답 도우미입니다."),
        ("human", "{question}")
    ]
)

# print("변수 목록 : ", prompt.input_variables)

# 입력을 dict 타입
value = prompt.invoke(
    {
        "question" : "부산 출장 일비는?"
    }
)
print(value)

for msg in value.to_messages():
    print(msg.content)

messages=[SystemMessage(content='당신은 사내 규정 질의 응답 도우미입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산 출장 일비는?', additional_kwargs={}, response_metadata={})]
당신은 사내 규정 질의 응답 도우미입니다.
부산 출장 일비는?


In [3]:
# prompt | llm | parser 연결 테스트
from langchain_core.language_models import FakeListChatModel, ParrotFakeChatModel
from langchain_core.output_parsers import StrOutputParser

# 앵무새 모델
# - 받은 메시지 그대로 리턴
echo = (prompt | ParrotFakeChatModel()).invoke(
    {
        "question" : "부산 출장 일비는?"
    }
)
print(echo.content)
print(type(echo).__name__)

chain = (
    prompt
    | FakeListChatModel(responses=["부산 출장 일비는 1일 4만원입니다."])
    | StrOutputParser()
)
print("체인 결과 : ", chain.invoke(
    {
        "question" : "부산 출장 일비는?"
    }
))

print("체인 타입 : ", type(chain).__name__)

부산 출장 일비는?
HumanMessage
체인 결과 :  부산 출장 일비는 1일 4만원입니다.
체인 타입 :  RunnableSequence


In [4]:
from pathlib import Path 

PROMPT_PATH = Path("backend", "app", "agent", "prompts", "answer_system.md")
raw = PROMPT_PATH.read_text(encoding="utf-8")

# 중괄호가 있으면 변수로 인식할 수 있어서 확인
print(f"answer_system.md : {len(raw.splitlines())}줄, 중괄호 {raw.count('{')}개, {raw.count('}')}개")

ChatPromptTemplate.from_messages(
    [
        ("system", raw),
        ("human", "{question}")
    ]
).invoke(
    {
        "question" : "부산 출장 일비는?"
    }
)
# 에러 없으면 통과됨
print("통과")

# --- 
# 중괄호 때문에 안되는 예

risky = raw + '\n\n예시: {"answer": "...", "sources": [{"doc_id" : "DOC-HR-011"}]}'
try:
        
    ChatPromptTemplate.from_messages(
        [
            ("system", risky),
            ("human", "{question}")
        ]
    ).invoke(
        {
            "question" : "부산 출장 일비는?"
        }
    )
except ValueError as e:
    print("예외 발생 ")
    print(e)

answer_system.md : 40줄, 중괄호 0개, 0개
통과
예외 발생 
Invalid format specifier in f-string template. Nested replacement fields are not allowed.


In [5]:
# 중괄호를 이스케이프 문자열로 처리?
# 이중 중괄호로 쓰기?
# System 메시지를 객체 타입으로 처리

from langchain_core.messages import SystemMessage

safe_prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=risky),
        ("human", "{question}")
    ]
)


val = safe_prompt.invoke(
    {
        "question": "부산 출장 일비는?"
    }
)

print(val)
print(safe_prompt.input_variables)

messages=[SystemMessage(content='## 역할\n\n한화시스템 임직원이 사내 규정에 관해 물으면, 함께 주어진 근거 문서만 읽고 답합니다.\n등록된 근거 문서는 다음 세 건입니다.\n\n| 문서 번호 | 제목 | 버전 | 소관 | 보안등급 |\n|---|---|---|---|---|\n| DOC-HR-014 | 국내출장 여비 규정 | v2.0 | 인사총무 | 일반 |\n| DOC-PU-007 | 구매·계약 규정 | v4.0 | 구매팀 | 대외비 |\n| DOC-SE-003 | 정보보안 지침 | v2.2 | 보안팀 | 대외비 |\n\n## 규칙\n\n1. 주어진 근거 문서 밖의 내용을 지어내지 않습니다. 모르면 모른다고 말합니다.\n2. 근거가 부족하면 부족하다고 먼저 밝힙니다. 다른 문서의 내용으로 유추해서 메우지 않습니다.\n3. 열람 권한이 없는 문서는 인용하지 않습니다. 보안등급은 일반·3급·대외비 세 단계입니다.\n4. 금액·기한·조건은 근거에 적힌 숫자를 그대로 옮깁니다. 계산이 필요하면 계산 과정을 보입니다.\n5. "아마", "일반적으로" 같은 표현으로 빈틈을 메우지 않습니다.\n\n## 출력 형식\n\n아래 세 필드를 가진 JSON 하나만 보냅니다.\n\n| 필드 | 형 | 뜻 |\n|---|---|---|\n| `answer` | 문자열 | 한국어 답변 본문. 결론을 먼저 씁니다 |\n| `sources` | 목록 | 인용한 근거. 항목마다 `doc_id` · `title` · `version` · `locator` |\n| `enough_evidence` | 참/거짓 | 근거가 충분했으면 true, 부족하면 false |\n\n- `doc_id` 는 `DOC-HR-014` · `DOC-PU-007` · `DOC-SE-003` 중 하나만 씁니다.\n  목록에 없는 문서 번호를 지어내지 않습니다.\n- 근거가 없으면 `sources` 를 빈 목록으로 두고 `enough_evidence` 를 false 로 둡니다.\n- 설명

* llm -> runnable

In [6]:
from langchain_core.runnables import Runnable, RunnableLambda

from app.integrations.ports import LLMResult, LLMPort

# Claude, OpenAI 등 LLM ... 
class MockLLM:
    def answer(
            self, 
            *, 
            question:str,
            contexts: list[dict],
            user: dict
        ) -> LLMResult: 
        # 가짜 LLM의 가짜 리턴값
        return LLMResult(
            text="부산 출장 일비는 1일 4만원입니다.",
            model="mock",
            input_tok=10,
            output_tok=12,
            cost_krw=0.0,
            latency_ms=1
        )

print(isinstance(MockLLM(), LLMPort))

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=raw),
        ("human", "{question}")
    ]
)

llm = MockLLM()

# ChatPromptTemplate 받아서 우리 포트를 부르는 연결 고리 생성
def call_port(value) -> LLMResult:
    msgs = value.to_messages()
    text = "\n\n".join(m.content for m in msgs)
    return llm.answer(question=text, contexts=[], user={})

# 체인에 연결 가능하게 runnable로 변환
llm_step = RunnableLambda(call_port).with_config(run_name="LLMPort")

chain = prompt | llm_step
result = chain.invoke(
    {
        "question": "부산 출장 일비는?"
    }
)
print("result.text : ", result.text)
print("result.model : ", result.model)


True
result.text :  부산 출장 일비는 1일 4만원입니다.
result.model :  mock


* parser -> StrOutputParser 

=> 우리가 만든 LLMResult 타입은 StrOutputParser가 인식하지 못함

In [7]:
# chain = prompt | llm_step | RunnableLambda(lambda r: r.text)

try:
    (prompt | llm_step | StrOutputParser()).invoke(
        {
            "question": "부산 출장 일비는?"
        }
    )
except Exception as e:
    print("StrOutputParser 붙였을 때 ...")
    print(e)

StrOutputParser 붙였을 때 ...
1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value=LLMResult(text='부산 ...latency_ms=1, extras={}), input_type=LLMResult]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


In [8]:
text_chain = prompt | llm_step | RunnableLambda(lambda r: r.text)
answer_txt = text_chain.invoke(
    {
        "question": "부산 출장 일비는?"
    }
)

print(answer_txt)

부산 출장 일비는 1일 4만원입니다.


* parser -> PydanticOutputParser

In [9]:
import json
from langchain_core.output_parsers import PydanticOutputParser
from app.schemas.chat import AnswerOut

parser = PydanticOutputParser(pydantic_object=AnswerOut)

print("형식 지시문 : ", parser.get_format_instructions().splitlines()[0])

# 질문
GOLDEN = {
    "answer": "부산 출장 일비는 1일 2만원입니다.",
    "sources": [
        {
            "doc_id" : "DOC-HR-014",
            "title": "국내출장 여비 규정",
            "version": "v2.0",
            "locator": "제12조 · p.6",
        }
    ],
    "enough_evidence": True,

}

# 가짜 LLM
class MockLLMJson:
    def answer(
            self,
            *,
            question: str,
            contexts: list[dict],
            user: dict
    ) -> LLMResult:
        return LLMResult(
            text=json.dumps(GOLDEN, ensure_ascii=False), model="mock"
        )

llm_json = MockLLMJson()
# Runnable로 변환 -> 파이프라인에 연결
llm_step_json = RunnableLambda(
    lambda value: llm_json.answer(
        question="\n\n".join(m.content for m in value.to_messages()),
        contexts=[],
        user={},
    )
).with_config(run_name="LLMPort")

# 파이프라인 생성
parsed_chain = prompt | llm_step_json | RunnableLambda(lambda r: r.text) | parser
parsed = parsed_chain.invoke(
    {
        "question" : "부산 출장 일비는?"
    }
)

print("결과 타입 : ", type(parsed).__name__)
print("answer : ", parsed.answer)
print("enough_evidence : ", parsed.enough_evidence)

형식 지시문 :  The output should be formatted as a JSON instance that conforms to the JSON schema below.
결과 타입 :  AnswerOut
answer :  부산 출장 일비는 1일 2만원입니다.
enough_evidence :  True


In [10]:
# json 타입이 아닌 가짜 LLM 
class MockLLMPlain:
    def answer(
            self,
            *,
            question: str,
            contexts: list[dict],
            user:dict
    ) -> LLMResult:
        return LLMResult(
            text="죄송합니다. 잘 모르겠습니다.",
            model="mock"
        )
llm_plain = MockLLMPlain()
plain_chain = (
    prompt
    | RunnableLambda(
        lambda value: llm_plain.answer(
            question="\n\n".join(m.content for m in value.to_messages()),
            contexts=[],
            user={},
        )
    )
    | RunnableLambda(lambda r: r.text)
    | parser
)

try:
    plain_chain.invoke(
        {
            "question" : "부산 출장 일비는?"
        }
    )
except Exception as exc:
    print("형식을 안 지켰을 때 :", type(exc).__name__)

형식을 안 지켰을 때 : OutputParserException


In [11]:
import importlib
import app.agent.chain as chain_module

importlib.reload(chain_module)

from app.agent.chain import build_answer_chain, build_parsed_chain, load_prompt

system_prompt = load_prompt("answer_system")
print("프롬프트 앞 부분 : ", system_prompt[:50])

text = build_answer_chain(MockLLM(), system_prompt=system_prompt).invoke(
    {
        "question": "부산 출장 일비는?"
    }
)

print("build_answer_chain : ", text)

parsed = build_parsed_chain(
    MockLLMJson(), schema=AnswerOut, system_prompt=system_prompt
).invoke(
    {
        "question" : "부산 출장 일비는?"
    }
)

print("parsed.answer : ", parsed.answer)
print("parsed.sources : ", parsed.sources[0].doc_id)


프롬프트 앞 부분 :  ## 역할

한화시스템 임직원이 사내 규정에 관해 물으면, 함께 주어진 근거 문서만 읽고 
build_answer_chain :  부산 출장 일비는 1일 4만원입니다.
parsed.answer :  부산 출장 일비는 1일 2만원입니다.
parsed.sources :  DOC-HR-014
